# Prepare LLM Annotation Batches

This notebook converts the LLM-training pool into compact JSONL batches for the annotator model. It does not call an LLM; it creates auditable files for Codex/OpenAI batch annotation through the chosen interface.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "01_classification":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

CLASSIFICATION_DIR = PROJECT_ROOT / "data/interim/lsc/classification"
LLM_DIR = CLASSIFICATION_DIR / "llm_annotation"
BATCH_DIR = LLM_DIR / "annotator_batches"
RAW_OUTPUT_DIR = LLM_DIR / "annotator_raw_outputs"
PARSED_DIR = LLM_DIR / "annotator_parsed"

for path in [BATCH_DIR, RAW_OUTPUT_DIR, PARSED_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LLM_POOL_PATH = LLM_DIR / "frame_llm_training_pool.csv"
PROMPT_PATH = PROJECT_ROOT / "notebooks/01_classification/prompts/annotator_v2.md"
BATCH_SIZE = 75
PROMPT_VERSION = "annotator_v2"

## Write Annotator Input Batches

Each row carries only the metadata needed for annotation and parsing. The prompt file is stored separately and should be pasted or supplied with each batch.

In [ ]:
pool = pd.read_csv(LLM_POOL_PATH)
pool = pool.reset_index(drop=True).copy()
pool["annotation_id"] = [f"llm_train_{i:05d}" for i in range(1, len(pool) + 1)]
pool["prompt_version"] = PROMPT_VERSION

batch_rows = []
for batch_index, start in enumerate(range(0, len(pool), BATCH_SIZE), start=1):
    batch = pool.iloc[start : start + BATCH_SIZE].copy()
    batch_name = f"annotator_batch_{batch_index:03d}.jsonl"
    batch_path = BATCH_DIR / batch_name
    with batch_path.open("w", encoding="utf-8") as handle:
        for _, row in batch.iterrows():
            payload = {
                "annotation_id": row["annotation_id"],
                "context_id": row["context_id"],
                "target": row["analysis_unit"],
                "raw_form": row["raw_form"],
                "passage": row["target_sentence_plus_adjacent"],
            }
            handle.write(json.dumps(payload, ensure_ascii=False) + "\n")
    batch_rows.append({"batch_name": batch_name, "rows": len(batch), "start_row": start, "end_row": start + len(batch) - 1})

manifest = pd.DataFrame(batch_rows)
manifest_path = LLM_DIR / "annotator_batch_manifest.csv"
pool_path = LLM_DIR / "frame_llm_training_pool_with_annotation_ids.csv"
manifest.to_csv(manifest_path, index=False)
pool.to_csv(pool_path, index=False)

print(f"Prompt: {PROMPT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Wrote {len(manifest):,} batches to {BATCH_DIR.relative_to(PROJECT_ROOT)}")

## Parse Annotator Outputs

After LLM annotation, save one raw JSONL output file per batch in `annotator_raw_outputs/` using the same batch filename. Rerun this section to combine outputs.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            text = line.strip()
            if not text:
                continue
            try:
                rows.append(json.loads(text))
            except json.JSONDecodeError as error:
                raise ValueError(f"Invalid JSON in {path.name} line {line_number}: {error}") from error
    return rows

required_output_columns = [
    "annotation_id",
    "substantive_target_discourse",
    "clinical_frame_present",
    "lived_experience_frame_present",
    "confidence",
]
raw_files = sorted(RAW_OUTPUT_DIR.glob("*.jsonl"))
if not raw_files:
    print(f"No raw annotator outputs found in {RAW_OUTPUT_DIR.relative_to(PROJECT_ROOT)} yet.")
else:
    parsed_rows = []
    for path in raw_files:
        for row in read_jsonl(path):
            row["source_file"] = path.name
            parsed_rows.append(row)
    parsed = pd.DataFrame(parsed_rows)
    missing = sorted(set(required_output_columns) - set(parsed.columns))
    if missing:
        raise ValueError(f"LLM output is missing required columns: {missing}")
    parsed_path = PARSED_DIR / "frame_llm_annotator_labels.csv"
    parsed.to_csv(parsed_path, index=False)
    print(f"Parsed {len(parsed):,} LLM annotation rows to {parsed_path.relative_to(PROJECT_ROOT)}")
